In [1]:
from onep import paths
#Plot shock-responsive cells only - revisions
import pickle
import numpy as np

with open(paths.collection("fc"), 'rb') as file:
    collection_fc = pickle.load(file)

def get_traces(collection, animals, stop_index=3303):
    return np.hstack([
        collection.animals[a].accepted_traces.to_numpy()[:stop_index, :]
        for a in animals
    ])

mouse_ids = ['astroF3', 'astroF5', 'astroF6', 'astroF7','astroF9', 'astroF10',
             'astroM3', 'astroM4', 'astroM5', 'astroM6', 'astroM7', 'astroM8', 'astroM9', 'astroM10']

fcA_array = get_traces(collection_fc, mouse_ids) #create array
print("fcA_array shape:", fcA_array.shape)  #(time indices, cells)

timestamps = collection_fc.animals['astroM4'].Timestamps[:3303].to_numpy().squeeze() #make timestamps array

fcA_array shape: (3303, 2823)


In [2]:
def detect_shock_responsive_cells_any_peak(
    data_z,
    shock_times,
    sampling_rate=10,
    baseline_window=5,
    response_window=15,
    threshold=1.96      #std above mean
):

    T, N = data_z.shape
    n_events = len(shock_times)

    peak_times  = np.zeros((n_events, N), dtype=float)
    peak_values = np.zeros((n_events, N), dtype=float)
    deltas      = np.zeros((n_events, N), dtype=float)

    for k, e in enumerate(shock_times):
        # indices
        sb = int((e - baseline_window) * sampling_rate)
        eb = int(e * sampling_rate)
        sr = int(e * sampling_rate)
        er = int((e + response_window) * sampling_rate)

        # non-empty slices
        sb = max(0, sb)
        eb = max(sb + 1, min(T, eb))
        sr = max(0, sr)
        er = max(sr + 1, min(T, er))

        baseline = data_z[sb:eb, :]
        response = data_z[sr:er, :]

        base_mean = baseline.mean(axis=0)
        p_idx = np.argmax(response, axis=0)
        p_vals = response[p_idx, np.arange(N)]
        p_times = (sr + p_idx) / float(sampling_rate)

        peak_values[k, :] = p_vals
        peak_times[k, :]  = p_times
        deltas[k, :]      = p_vals - base_mean   # peak - baseline (mean) = delta

    responsive_mask_per_event = deltas > threshold #if delta greater than STD threshold, make binary for each event
    responsive_any = responsive_mask_per_event.any(axis=0)
    responsive_cells = np.where(responsive_any)[0]
    non_responsive_cells = np.setdiff1d(np.arange(N), responsive_cells)

    return (responsive_cells,
            non_responsive_cells,
            peak_times,
            peak_values,
            responsive_mask_per_event,
            deltas)

In [3]:
import matplotlib.pyplot as plt

def plot_individual_cells_with_peaks(
    data_z,
    shock_times,
    responsive_cells,
    non_responsive_cells,
    peak_times,
    peak_values,
    responsive_mask_per_event,
    sampling_rate=10,
    max_cells=100
):
    T, N = data_z.shape
    time = np.arange(T) / float(sampling_rate)
    cells_to_plot = min(max_cells, N)

    fig, axs = plt.subplots(cells_to_plot, 1, figsize=(6, 0.6*cells_to_plot), sharex=True)
    if cells_to_plot == 1:
        axs = [axs]

    resp_set = set(responsive_cells)

    for i in range(cells_to_plot):
        ax = axs[i]
        trace_color = "#B39BC8" if i in resp_set else "gray"
        ax.plot(time, data_z[:, i], color=trace_color, linewidth=0.8)

        # shock markers
        for s in shock_times:
            ax.axvline(s, color='red', linestyle='--', linewidth=0.6)

        # peak markers per event (green if event-responsive, light gray otherwise)
        for k in range(len(shock_times)):
            ax.plot(peak_times[k, i], peak_values[k, i],
                    marker='o',
                    markersize=3.5,
                    mec='black',
                    mfc=("#31a354" if responsive_mask_per_event[k, i] else "#bdbdbd"),
                    linewidth=0)

        ax.axhline(0, color='black', linestyle=':', linewidth=0.5)
        ax.set_yticks([])
        ax.set_ylabel(f"{i}", rotation=0, labelpad=10, fontsize=6, va='center')
        ax.spines[['top','right','left']].set_visible(False)

    axs[-1].set_xlabel("Time (s)")
    plt.suptitle(
        "Individual Cells (purple = any-shock responsive)\n",
        fontsize=10, y=1.01
    )
    plt.tight_layout()
    plt.show()

In [4]:
from scipy import stats

fcA_array_z = stats.zscore(fcA_array, axis=0) #zsscore your data column-wise

shock_times = [120, 180, 240, 300] #set your shock times

(responsive_cells,non_responsive_cells,peak_times,peak_values,resp_mask,deltas) = detect_shock_responsive_cells_any_peak(
    fcA_array_z,
    shock_times=shock_times,
    sampling_rate=10, #10hz
    baseline_window=5, #5 sec pre
    response_window=10, #10 sec post
    threshold=1.96 #95% ci
)

print(f"Responsive to ANY shock): {len(responsive_cells)}")
print(f"Non-responsive: {len(non_responsive_cells)}")

plot_individual_cells_with_peaks(
    fcA_array_z,
    shock_times,
    responsive_cells,
    non_responsive_cells,
    peak_times,
    peak_values,
    resp_mask,
    sampling_rate=10,
    max_cells=10
)

Responsive to ANY shock): 2766
Non-responsive: 57


C:\Users\ryansenne\AppData\Local\Temp\ipykernel_14832\3822251968.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

data = [2766,57]

labels = ['Shock-Responsive (>= 1 Shock)', 'Non-Responsive']

colors = sns.color_palette('Set3')

fig1, ax1 = plt.subplots()
ax1.pie(data,
        labels=labels,
        colors=colors,
        textprops = {'fontsize': 14},
        autopct='%1.1f%%', # Format for displaying percentages
        shadow=False,
        startangle=45)

ax1.axis('equal')
plt.tight_layout()
plt.savefig(paths.figure_path("figure1", "shock_responsive_piechart.svg"))
plt.show()

# Optional: save to file


C:\Users\ryansenne\AppData\Local\Temp\ipykernel_14832\3833350194.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
